# 02 — Data Cleaning
Olist E-Commerce Python Analytics

Cleans all 9 raw tables based on findings from `01_raw_data_exploration.ipynb`. Each table gets
its own section with the specific fixes it needs — no table is forced through a generic
one-size-fits-all cleaning loop, since most real issues here needed individual judgment calls,
not blanket rules.

**Convention:** raw files are copied once into `data/cleaned/` as untouched working copies, then
modified in place throughout this notebook and overwritten at the end.

For each step: the question/decision being addressed is in a `# Q:` comment, and what was found
or decided is in a `# Finding:` / `# Decision:` / `# Fix:` comment placed right below.

## 1. Setup — imports, folders, working copies

In [1]:
import pandas as pd
import numpy as np
import os
import shutil

# Load raw (untouched) versions once — used throughout for raw-vs-clean comparisons
raw_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
raw_orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
raw_order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
raw_order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
raw_order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
raw_products = pd.read_csv('../data/raw/olist_products_dataset.csv')
raw_category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
raw_sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
raw_geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

In [2]:
# Q: Where should cleaning happen -- directly on the raw files, or on separate working copies?
# Decision: copy raw CSVs into data/cleaned/ ONCE as untouched working copies, modify those
# in place. Re-running this cell after cleaning has started would wipe out in-progress work,
# so don't re-execute it later.
os.makedirs('../data/cleaned', exist_ok=True)

files = [
    'olist_customers_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv',
    'product_category_name_translation.csv', 'olist_sellers_dataset.csv', 'olist_geolocation_dataset.csv'
]
for f in files:
    shutil.copy(f'../data/raw/{f}', f'../data/cleaned/{f}')

# Working copies — these get modified throughout the notebook, dropping the raw_ prefix
customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv')
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/cleaned/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/cleaned/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/cleaned/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/cleaned/product_category_name_translation.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv')
geolocation = pd.read_csv('../data/cleaned/olist_geolocation_dataset.csv')

## 2. Quick refresher — shape / dtypes / nulls across all tables
Not new analysis — just a fast glance before diving into per-table cleaning, since the full
audit already happened in notebook 01.

In [3]:
# Q: Any quick, fast confirmation that every table loaded as expected before starting?
tables = {
    'customers': customers, 'orders': orders, 'order_items': order_items,
    'order_payments': order_payments, 'order_reviews': order_reviews, 'products': products,
    'category_translation': category_translation, 'sellers': sellers, 'geolocation': geolocation
}

for name, df in tables.items():
    print(f"{name:22s} shape={df.shape}  nulls={df.isna().sum().sum()}")
# Finding: matches the 01 audit exactly -- no surprises. This is just a sanity glance, not
# new analysis.

customers              shape=(99441, 5)  nulls=0
orders                 shape=(99441, 8)  nulls=4908
order_items            shape=(112650, 7)  nulls=0
order_payments         shape=(103886, 5)  nulls=0
order_reviews          shape=(99224, 7)  nulls=145903
products               shape=(32951, 9)  nulls=2448
category_translation   shape=(71, 2)  nulls=0
sellers                shape=(3095, 4)  nulls=0
geolocation            shape=(1000163, 5)  nulls=0


## 3. Customers
Cleanest table in the dataset — only real fixes are text normalization on city names and
zero-padding the zip prefix (zero-padding is applied together with sellers/geolocation in
section 12, so all three zip columns change in the same step).

In [4]:
# Q: What does the customers table actually need fixed, given the raw audit found it almost
# entirely clean?
# Context: customer_id (per-order surrogate) vs customer_unique_id (real person) -- audit found
# 96,096 unique people across 99,441 rows. ALWAYS use customer_unique_id for repeat-customer /
# RFM analysis later.
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
customers['customer_city'].nunique()  # confirm normalization collapsed casing/whitespace duplicates
# Finding: strip+lower normalization is the only fix this table needs -- no nulls, no
# duplicates, no invalid values to correct.

4119

## 4. Orders
The heaviest cleaning in the project — date conversion, two genuine date-contradiction
anomalies (flagged AND nulled, not silently dropped), and derived delivery metrics.

In [5]:
# Q: Does converting all 5 date columns to real datetimes introduce any NEW nulls that
# weren't already there as blank strings?
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']

null_before = orders[date_cols].isna().sum()
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])
null_after = orders[date_cols].isna().sum()

assert (null_before == null_after).all(), 'Datetime conversion introduced new nulls — investigate!'
# Finding: null counts identical before/after conversion -- clean conversion, nothing hidden.

In [6]:
# Q: How should "late delivery" be defined and labeled?
# Decision: this is a business-meaningful OUTCOME, not a data error -- a late delivery is a
# real, valid result (the headline Fulfillment Intelligence metric), so it's never "fixed",
# only labeled.
def lateDelivery(row):
    if pd.isna(row['order_delivered_customer_date']) or pd.isna(row['order_estimated_delivery_date']):
        return 'Unknown'
    if row['order_delivered_customer_date'] > row['order_estimated_delivery_date']:
        return 'Late Delivery'
    return 'On Time Delivery'

orders['lateDelivery'] = orders.apply(lateDelivery, axis=1)
orders['lateDelivery'].value_counts()
# Finding: matches the raw audit's 7,827-late-order figure (will be rebuilt below after
# nulling, since 23 rows need to shift to 'Unknown').

lateDelivery
On Time Delivery    88649
Late Delivery        7827
Unknown              2965
Name: count, dtype: int64

In [7]:
# Q: How should the two genuine date-order contradictions found in the raw audit be flagged,
# without conflating real errors with normal system-logging timing noise?
# Decision: 'Approved After Carrier Delivered' uses a 2-day cutoff (chosen from the gap
# distribution: median ~17hrs, 75th pct ~1 day) to separate harmless logging noise from real
# errors -- WITHOUT the cutoff this would wrongly flag ~1,359 rows instead of the genuine ~176.
# 'Carrier After Customer' (23 rows) is always an error -- physically impossible, no cutoff needed.
cutoff = pd.Timedelta(days=2)

def orderCheck(row):
    if pd.isna(row['order_delivered_customer_date']) or pd.isna(row['order_delivered_carrier_date']) or pd.isna(row['order_approved_at']):
        return 'Unknown'
    if (row['order_approved_at'] - row['order_delivered_carrier_date']) > cutoff:
        return 'Approved After Carrier Delivered'
    elif row['order_delivered_carrier_date'] > row['order_delivered_customer_date']:
        return 'Carrier After Customer'
    return 'Valid'

orders['validDateCheck'] = orders.apply(orderCheck, axis=1)
orders['validDateCheck'].value_counts()
# Finding: cutoff correctly isolates the genuine error population from normal noise.

validDateCheck
Valid                               96262
Unknown                              2980
Approved After Carrier Delivered      176
Carrier After Customer                 23
Name: count, dtype: int64

In [8]:
# Q: Does the if/elif branching in orderCheck hide any row that technically satisfies BOTH
# violation conditions at once?
overlap_mask = (
    ((orders['order_approved_at'] - orders['order_delivered_carrier_date']) > cutoff) &
    (orders['order_delivered_carrier_date'] > orders['order_delivered_customer_date'])
)
print(f"Rows satisfying both violation conditions: {overlap_mask.sum()}")
orders[overlap_mask]
# Finding: 0 rows -- confirmed the single-label flag isn't silently hiding a double-violation.

Rows satisfying both violation conditions: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,lateDelivery,validDateCheck


In [9]:
# Q: Once flagged, how should the contradictory rows actually be fixed -- drop them, leave
# them, or something in between?
# Decision: flag + null -- keep the flag column for transparency/auditability, but also null
# the SPECIFIC contradictory date fields so downstream aggregates (mean delivery time, etc.)
# can't be silently corrupted by impossible dates. Relying on "remember to filter by flag
# everywhere" is fragile.
orders.loc[orders['validDateCheck'] == 'Approved After Carrier Delivered',
           ['order_approved_at', 'order_delivered_carrier_date']] = pd.NaT
orders.loc[orders['validDateCheck'] == 'Carrier After Customer',
           ['order_delivered_carrier_date', 'order_delivered_customer_date']] = pd.NaT

# Verify the nulls landed exactly where the flag says they should
orders.groupby('validDateCheck')[['order_approved_at','order_delivered_carrier_date',
                                    'order_delivered_customer_date']].apply(lambda x: x.isna().sum())
# Finding: nulls land exactly where expected per flag category -- the untouched field in each
# anomaly group remains populated.

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
validDateCheck,,,
Approved After Carrier Delivered,176,176,0
Carrier After Customer,0,23,23
Unknown,160,1783,2965
Valid,0,0,0


In [10]:
# Q: Does lateDelivery need to be recomputed after the nulling above?
# Fix: yes -- the 23 'Carrier After Customer' rows must now correctly fall to 'Unknown' since
# their delivery date was just nulled, not read from a stale pre-nulling calculation.
orders['lateDelivery'] = orders.apply(lateDelivery, axis=1)

In [11]:
# Q: What are the headline delivery-timing metrics needed for Fulfillment Intelligence?
# Decision: NaN automatically for any row with a null date -- correct, since you can't compute
# a delivery time for an order that has none logged.
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

orders[['delivery_days','delivery_delay_days']].describe()
# Finding: both derived columns behave as expected -- no negative/impossible values, nulls
# line up with missing dates.

,delivery_days,delivery_delay_days
count,96453.000000,96453.000000
mean,12.095435,-11.873876
std,9.552181,10.181480
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


**Known, documented anomaly (not fixed, just noted):** 1 order (status `shipped`) has no
`order_items`, no carrier date, no customer date — a genuine incomplete record. Kept in
order-count totals, naturally excluded from delivery/item metrics since its dates are null.

## 5. Order Items

In [12]:
# Q: Does shipping_limit_date need conversion to datetime?
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [13]:
# Q: How should extreme freight/price combinations (freight > price, freight == 0) be
# flagged -- assumed to be errors, or verified first?
# Decision: flag both extremes for later verification, not assumed to be data errors --
# 'Delivery Expensive' and 'Free Delivery' are both checked against product/seller patterns
# below before being trusted as legitimate business patterns.
def priceFreightCheck(row):
    if pd.isna(row['freight_value']) or pd.isna(row['price']):
        return 'Unknown'
    elif row['freight_value'] > row['price']:
        return 'Delivery Expensive'
    elif row['freight_value'] == 0:
        return 'Free Delivery'
    return 'Normal'

order_items['priceFreightCheck'] = order_items.apply(priceFreightCheck, axis=1)
order_items['priceFreightCheck'].value_counts()

priceFreightCheck
Normal                108143
Delivery Expensive      4124
Free Delivery            383
Name: count, dtype: int64

In [14]:
# Q: Is the 'Free Delivery' pattern spread randomly across sellers, or concentrated?
order_items[order_items['priceFreightCheck']=='Free Delivery']['seller_id'].value_counts()
# Finding: confirmed concentrated in just 9 sellers (96% from top 4) -- a seller promotion
# pattern, not random missing data.

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64

In [15]:
# Q: Is 'Delivery Expensive' actually concentrated in cheap/bulky items (the assumed story),
# or spread evenly across categories and price points (which would need a different
# explanation before being trusted as a legitimate pattern)?
expensive = order_items[order_items['priceFreightCheck'] == 'Delivery Expensive'] \
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')

print(expensive['product_category_name'].value_counts(normalize=True).head(10))
print(expensive['price'].describe())
# Finding: confirmed concentrated in cheap/bulky categories and low price points -- treated as
# a legitimate business pattern, not a data error.

product_category_name
eletronicos               0.151425
telefonia                 0.106568
utilidades_domesticas     0.095415
moveis_decoracao          0.082776
beleza_saude              0.074102
esporte_lazer             0.054771
cama_mesa_banho           0.050558
automotivo                0.043123
informatica_acessorios    0.043123
perfumaria                0.035688
Name: proportion, dtype: float64
count    4124.000000
mean       17.623378
std        15.994419
min         0.850000
25%        10.500000
50%        13.900000
75%        18.900000
max       189.900000
Name: price, dtype: float64


In [16]:
# Q: What single combined-cost figure is useful for revenue analysis downstream?
# Decision: deliberately NOT called 'profit' or 'revenue' -- Olist gives no COGS data, so this
# is just the combined charge to the customer for one line item (price + shipping), nothing more.
order_items['item_total_value'] = order_items['price'] + order_items['freight_value']

## 6. Payments
Many-to-one with orders (1–5 payment rows per order) — aggregation to order level is deferred
to the analytical-dataset stage, not done here.

In [17]:
# Q: Are all $0 payment rows the same kind of issue?
# Decision: no -- zero_value_flag distinguishes two very different $0 patterns found in the
# audit: voucher rows (6) are legitimate (a voucher fully covered the order); not_defined rows
# (3) are genuinely broken records (unknown type AND no value). Treating both the same would
# hide this difference.
order_payments['zero_value_flag'] = np.where(
    (order_payments['payment_value']==0) & (order_payments['payment_type']=='not_defined'), 'Broken Record',
    np.where((order_payments['payment_value']==0) & (order_payments['payment_type']=='voucher'),
             'Voucher Full Coverage', 'Normal')
)
order_payments['zero_value_flag'].value_counts()
# Finding: matches the raw audit exactly -- 3 Broken Record, 6 Voucher Full Coverage.

zero_value_flag
Normal                   103877
Voucher Full Coverage         6
Broken Record                 3
Name: count, dtype: int64

In [18]:
# Q: Is payment_installments == 0 ever a valid real-world value?
# Decision: no -- both rows are real credit_card payments with nonzero values, a genuine
# data-entry error. Corrected to 1 (not just flagged), since 0 installments has no valid
# meaning and the correct value is obvious.
order_payments.loc[order_payments['payment_installments']==0, 'payment_installments'] = 1
(order_payments['payment_installments']==0).sum()  # should be 0
# Finding: 0 remaining -- fix confirmed.

np.int64(0)

## 7. Payments vs Order Items — cross-table reconciliation
Every check so far has validated a table against itself. This is the one check that spans two
independently-collected tables: does what the customer was charged (`order_items`) match what
they actually paid (`order_payments`)? A mismatch here is a different, more serious class of
error than anything a single-table null/dtype/duplicate check can catch.

In [19]:
# Q: Does what the customer was charged (order_items) match what they actually paid
# (order_payments), at the per-order level?
order_totals = order_items.groupby('order_id')['item_total_value'].sum().rename('items_total')
payment_totals = order_payments.groupby('order_id')['payment_value'].sum().rename('payments_total')

reconciliation = order_totals.to_frame().join(payment_totals, how='outer')
reconciliation['diff'] = reconciliation['payments_total'] - reconciliation['items_total']

mismatches = reconciliation[reconciliation['diff'].abs() > 0.01]
print(f"Orders with payment/items mismatch: {len(mismatches)}")
mismatches.sort_values('diff', ascending=False).head(20)
# Finding: a real, non-zero mismatch count exists -- first time a cross-table (not
# single-table) check surfaces a different class of error.

Orders with payment/items mismatch: 378


,items_total,payments_total,diff
order_id,,,
ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81
6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01
70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89
996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53
70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69
bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01
af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80
bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43
8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44


## 8. Reviews
Has a genuine many-to-many relationship with orders — resolved into a separate
one-row-per-order lookup table so no future merge can silently duplicate rows.

In [20]:
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

# Q: Do whitespace-only comment strings ('   ') pass as real content?
# Fix: normalize to true NaN so any 'has a comment' flag or text-length analysis later isn't
# thrown off by phantom text -- these pass isna() as non-null but carry no real content.
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].replace(r'^\s*$', np.nan, regex=True)
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].replace(r'^\s*$', np.nan, regex=True)

In [21]:
# Q: Do the raw audit's sanity checks still return empty on this cleaned copy?
order_reviews[order_reviews['review_creation_date'] > order_reviews['review_answer_timestamp']]
order_reviews[(order_reviews['review_score']<1) | (order_reviews['review_score']>5)]
# Finding: both empty, as expected -- nothing in cleaning should have introduced these.

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [22]:
# Q: What response-time cutoff separates genuinely slow review responses from normal
# day-to-day variation?
# Decision: 10-day cutoff, chosen from the actual gap distribution (median ~1.7 days, 75th pct
# ~3 days) -- captures only the genuine long tail (~3% of reviews), not normal noise.
response_time = order_reviews['review_answer_timestamp'] - order_reviews['review_creation_date']
order_reviews['slow_response'] = np.where(response_time > pd.Timedelta(days=10), 'Slow Response', 'Normal')
order_reviews['slow_response'].value_counts()

slow_response
Normal           96252
Slow Response     2972
Name: count, dtype: int64

In [23]:
# Q: How should the many-to-many review_id <-> order_id relationship be collapsed to one
# row per order, without losing information a simple average would hide?
# Decision: aggregate to ONE row per order_id, keeping std alongside the mean so a volatile
# order (scores 1,5 -> avg 3) is distinguishable from a consistently mediocre one (scores 3,3
# -> avg 3) -- averaging alone would hide that difference.
# RULE: ALWAYS join this table for order-level review analysis, never the raw order_reviews directly.
order_level_reviews = order_reviews.sort_values('review_creation_date').groupby('order_id').agg(
    review_score_avg=('review_score', 'mean'),
    review_score_count=('review_score', 'count'),
    review_score_std=('review_score', 'std'),
    review_comment_title=('review_comment_title', 'last'),
    review_comment_message=('review_comment_message', 'last')
).reset_index()

order_level_reviews.shape  # expect (98673, 6) — matches unique order_id count from the raw audit
# Finding: shape matches -- no order lost or duplicated in the collapse.

(98673, 6)

In [24]:
# Q: What should a null review_score_std (single-review orders) actually mean?
# Decision: NaN std means "only one review, no variance to measure" -- not "unknown" -- so
# fill with 0 rather than leaving it null, since most orders have exactly one review and would
# otherwise carry a null std into every downstream aggregate.
order_level_reviews['review_score_std'] = order_level_reviews['review_score_std'].fillna(0)

## 9. Products

In [25]:
# Q: Are the source column names usable as-is?
# Fix: correct the dataset's own spelling errors for cleanliness/readability downstream.
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

In [26]:
# Q: Are the 610 metadata-missing rows and the 2 dimension-missing rows the same issue,
# or two different problems needing different handling?
# Decision: kept as TWO SEPARATE flags, not merged into one 'broken' label --
# listing_metadata_missing (610 rows) = no category/title/desc/photos, but dimensions ARE
# intact, so still usable for shipping/freight analysis (just needs an 'Unknown category'
# bucket elsewhere). dimensions_missing (2 rows) = no weight/size -- these actually break
# freight calculations.
products['listing_metadata_missing'] = products[['product_category_name','product_name_length',
    'product_description_length','product_photos_qty']].isna().all(axis=1)
products['dimensions_missing'] = products[['product_weight_g','product_length_cm',
    'product_height_cm','product_width_cm']].isna().any(axis=1)

products['listing_metadata_missing'].sum(), products['dimensions_missing'].sum()  # expect 610, 2
# Finding: matches the raw audit exactly.

(np.int64(610), np.int64(2))

In [27]:
# Q: How should the 4 physically-impossible product_weight_g == 0 rows be fixed?
# Decision: impute with that category's median, computed from valid NONZERO weights only
# (so the broken rows don't pollute their own fix) -- all 4 rows are in the same category
# (cama_mesa_banho) with real dimensions otherwise present.
median_weight = products[(products['product_category_name']=='cama_mesa_banho')
                          & (products['product_weight_g']>0)]['product_weight_g'].median()
products.loc[products['product_weight_g']==0, 'product_weight_g'] = median_weight

In [28]:
# Q: What happens to the 2 categories with no English translation (13 products affected)?
# Decision: manually mapped rather than left to produce silent nulls on merge.
manual_translations = {
    'pc_gamer': 'gaming_pc',
    'portateis_cozinha_e_preparadores_de_alimentos': 'kitchen_appliances_and_food_preparers'
}
new_rows = pd.DataFrame(list(manual_translations.items()),
                         columns=['product_category_name', 'product_category_name_english'])
category_translation = pd.concat([category_translation, new_rows], ignore_index=True)

products = products.merge(category_translation, on='product_category_name', how='left')
# Q: Did the merge introduce any NEW nulls beyond the already-known missing-metadata rows?
products['product_category_name_english'].isna().sum()
# Finding: null count equals exactly listing_metadata_missing's count (610) -- no new nulls
# introduced by the merge.

np.int64(610)

## 10. Sellers
Audit confirmed no nulls, no duplicates, no blank strings, sane zip range, and city text
already collapses cleanly under strip+lower — except one row where `seller_city` is a zip
code, not a city name.

In [29]:
# Q: Is the one seller_city containing digits ("04482255") a real city name, or a data error?
digit_city_mask = sellers['seller_city'].str.contains(r'\d', regex=True)
bad_row = sellers[digit_city_mask]
print(bad_row)
# Finding: confirmed a zip code sitting in the city field, not a real city -- flagged for
# recovery below.

                            seller_id  seller_zip_code_prefix seller_city  \
517  ceb7b4fb9401cd378de7886317ad1b47                   22790    04482255   

    seller_state  
517           RJ  


In [30]:
# Q: Can the real city for this row be recovered rather than just dropped or left as Unknown?
# Decision: recover via that seller's own zip prefix, looked up directly against
# raw_geolocation (geo_lookup doesn't exist yet at this point in the notebook -- it's built in
# section 11).
recovered = raw_geolocation.loc[
    raw_geolocation['geolocation_zip_code_prefix'] == bad_row['seller_zip_code_prefix'].values[0],
    'geolocation_city'
].mode()

sellers.loc[digit_city_mask, 'seller_city'] = recovered[0] if not recovered.empty else np.nan
sellers.loc[digit_city_mask]  # confirm the fix landed
# Finding: real city recovered and confirmed -- not itself a stray/garbage value.

,seller_id,seller_zip_code_prefix,seller_city,seller_state
517,ceb7b4fb9401cd378de7886317ad1b47,22790,rio de janeiro,RJ


## 11. Geolocation
~1M rows with no natural primary key. Never merged raw — always collapsed to a
one-row-per-zip-prefix lookup table first.

In [31]:
# Q: Are the 8,556 same-zip-different-city rows real geographic conflicts, or just text/
# encoding noise?
city_conflicts = raw_geolocation.groupby('geolocation_zip_code_prefix')['geolocation_city'].nunique()
(city_conflicts > 1).sum()
# Decision: investigated and confirmed pure accent-encoding noise ('sao paulo' vs 'sao paulo')
# -- not worth fixing directly (city text isn't critical to this table's purpose). Parked as a
# future enhancement (accent-stripping); resolved for now via modal aggregation below, which
# already picks the more frequent spelling per prefix without extra code.

np.int64(8556)

In [32]:
# Q: Are the full-row duplicates genuinely redundant, safe to drop?
geo_clean = raw_geolocation.drop_duplicates()
# Finding: 261,831 rows dropped -- genuinely redundant, no information lost.

In [33]:
# Q: Are all lat/lng values actually within Brazil, or does a generic +/-90/+/-180 check miss
# anything?
# Decision: use Brazil's ACTUAL bounding box (not the generic lat/lng limits) -- catches 8 rows
# with impossible coordinates (e.g. lat 45.07, lng 121.11 -- nowhere near Brazil) that a
# generic range check would completely miss.
geo_clean = geo_clean[
    (geo_clean['geolocation_lat'] >= -33.75) & (geo_clean['geolocation_lat'] <= 5.27) &
    (geo_clean['geolocation_lng'] >= -73.99) & (geo_clean['geolocation_lng'] <= -34.79)
]

In [34]:
# Q: Do any zip prefixes span more than one distinct STATE? (This matters more than the
# city conflict, because geo_lookup takes the MEAN of lat/lng, and a mean across two different
# states is meaningless -- unlike a mode-resolved city label, which just picks a spelling.)
state_conflicts = geo_clean.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique()
n_state_conflicts = (state_conflicts > 1).sum()
print(f"Prefixes with conflicting states: {n_state_conflicts}")

Prefixes with conflicting states: 8


In [35]:
# Decision: keep only rows matching each prefix's MODAL (majority) state before averaging
# coordinates, so a handful of mislabeled rows can't drag the mean point into the wrong state.
modal_state = geo_clean.groupby('geolocation_zip_code_prefix')['geolocation_state'] \
    .agg(lambda x: x.mode()[0]).rename('modal_state')
geo_clean = geo_clean.merge(modal_state, on='geolocation_zip_code_prefix')
geo_clean = geo_clean[geo_clean['geolocation_state'] == geo_clean['modal_state']].drop(columns='modal_state')

In [36]:
# Q: What does the final one-row-per-zip-prefix lookup table look like?
# Decision: collapse to ONE row per zip prefix -- this, never the raw table, is what gets
# joined to customers/sellers later. mode() for city/state naturally resolves any conflicting
# labels per prefix (accent noise or otherwise) by picking the most frequent one.
geo_lookup = geo_clean.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean'),
    city=('geolocation_city', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    state=('geolocation_state', lambda x: x.mode()[0] if not x.mode().empty else np.nan)
).reset_index()

geo_lookup.shape  # 19,010 — 5 fewer than raw's 19,015 unique prefixes, see note below
# Finding: 5 prefixes lost -- investigated below before accepting this as negligible.

(19010, 5)

In [37]:
# Q: What happened to the 5 "lost" prefixes -- did they lose ALL their rows to the
# bounding-box filter, and does that have any real-world impact on customers/sellers?
raw_prefixes = set(raw_geolocation['geolocation_zip_code_prefix'].unique())
clean_prefixes = set(geo_lookup['geolocation_zip_code_prefix'].unique())
lost_prefixes = raw_prefixes - clean_prefixes

n_customers_affected = raw_customers[raw_customers['customer_zip_code_prefix'].isin(lost_prefixes)].shape[0]
n_sellers_affected = raw_sellers[raw_sellers['seller_zip_code_prefix'].isin(lost_prefixes)].shape[0]
print(f"Lost prefixes: {lost_prefixes}")
print(f"Customers affected: {n_customers_affected}, Sellers affected: {n_sellers_affected}")
# Finding: 1 customer, 0 sellers affected -- negligible, documented, not worth further action.

Lost prefixes: {np.int64(18243), np.int64(53990), np.int64(78131), np.int64(83252), np.int64(95130)}
Customers affected: 1, Sellers affected: 0


## 12. Zip code prefix normalization
`customer_zip_code_prefix`, `seller_zip_code_prefix`, and `geolocation_zip_code_prefix` all
load as `int64`, which silently drops leading zeros (e.g. a real prefix of `01001` becomes
`1001`). These are categorical codes, not real numbers, so they're converted to fixed-width
strings here — applied to all three at once so joins between them keep working (an int on one
side and a zero-padded string on the other would silently fail to match).

In [38]:
# Q: Does loading zip prefixes as int64 lose real information?
# Finding: yes -- leading zeros silently dropped on real 5-digit prefixes.
# Fix: convert all three zip columns to zero-padded strings TOGETHER, in the same step, so no
# join between them can break from a dtype mismatch.
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
geo_lookup['geolocation_zip_code_prefix'] = geo_lookup['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# spot-check the padding landed correctly
print(customers['customer_zip_code_prefix'].head())
print(sellers['seller_zip_code_prefix'].head())
print(geo_lookup['geolocation_zip_code_prefix'].head())

0    14409
1    09790
2    01151
3    08775
4    13056
Name: customer_zip_code_prefix, dtype: str
0    13023
1    13844
2    20031
3    04195
4    12914
Name: seller_zip_code_prefix, dtype: str
0    01001
1    01002
2    01003
3    01004
4    01005
Name: geolocation_zip_code_prefix, dtype: str


## 13. Save all cleaned tables
Final checkpoint for this notebook — everything downstream reads from `data/cleaned/`, never
`data/raw/`.

In [39]:
customers.to_csv('../data/cleaned/olist_customers_dataset.csv', index=False)
orders.to_csv('../data/cleaned/olist_orders_dataset.csv', index=False)
order_items.to_csv('../data/cleaned/olist_order_items_dataset.csv', index=False)
order_payments.to_csv('../data/cleaned/olist_order_payments_dataset.csv', index=False)
order_reviews.to_csv('../data/cleaned/olist_order_reviews_dataset.csv', index=False)
order_level_reviews.to_csv('../data/cleaned/order_level_reviews.csv', index=False)
products.to_csv('../data/cleaned/olist_products_dataset.csv', index=False)
category_translation.to_csv('../data/cleaned/product_category_name_translation.csv', index=False)
sellers.to_csv('../data/cleaned/olist_sellers_dataset.csv', index=False)
geo_lookup.to_csv('../data/cleaned/geo_lookup.csv', index=False)

## 14. Closing checkpoint — raw vs. cleaned reconciliation
Quick type-wise sweep confirming every intended change actually landed, before moving to
`03_data_validation.ipynb` for the full formal validation pass.

In [40]:
# Q: Did row counts change where they should have (geolocation only), and stay stable
# everywhere else?
raw_row_counts = {
    'customers': len(raw_customers), 'orders': len(raw_orders), 'order_items': len(raw_order_items),
    'order_payments': len(raw_order_payments), 'order_reviews': len(raw_order_reviews),
    'products': len(raw_products), 'sellers': len(raw_sellers),
    'geolocation': len(raw_geolocation)  # expect a large drop here — duplicates + bad coords removed
}
clean_row_counts = {
    'customers': len(customers), 'orders': len(orders), 'order_items': len(order_items),
    'order_payments': len(order_payments), 'order_reviews': len(order_reviews),
    'products': len(products), 'sellers': len(sellers),
    'geolocation': len(geo_clean)  # post-filter, pre-aggregation row count
}
for name in raw_row_counts:
    print(f"{name:16s} raw={raw_row_counts[name]:>8}  clean={clean_row_counts[name]:>8}")
# Finding: every table except geolocation shows raw == clean row count, confirming cleaning
# never drops/adds rows anywhere except the deliberate geolocation filtering.

customers        raw=   99441  clean=   99441
orders           raw=   99441  clean=   99441
order_items      raw=  112650  clean=  112650
order_payments   raw=  103886  clean=  103886
order_reviews    raw=   99224  clean=   99224
products         raw=   32951  clean=   32951
sellers          raw=    3095  clean=    3095
geolocation      raw= 1000163  clean=  738291


In [41]:
# Q: Did each specific, individual correction actually take effect?
print('payment_installments == 0:', (order_payments['payment_installments']==0).sum(), '(expect 0)')
print('product_weight_g == 0:', (products['product_weight_g']==0).sum(), '(expect 0)')
print('seller_city with digits:', sellers['seller_city'].str.contains(r'\d', regex=True).sum(), '(expect 0)')
print('order_date_quality != Valid, dates still populated:',
      orders.loc[orders['validDateCheck'].isin(['Approved After Carrier Delivered','Carrier After Customer']),
                 ['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date']].notna().sum().sum(),
      '(expect low — only the untouched fields in each anomaly group remain populated)')
print('orders/items/payments reconciliation mismatches:', len(mismatches))
# Finding: all expected-zero checks pass. The notebook is ready to hand off to
# 03_data_validation.ipynb for a formal, independent validation pass.

payment_installments == 0: 0 (expect 0)
product_weight_g == 0: 0 (expect 0)
seller_city with digits: 0 (expect 0)
order_date_quality != Valid, dates still populated: 199 (expect low — only the untouched fields in each anomaly group remain populated)
orders/items/payments reconciliation mismatches: 378
